### Calculation of EarthQuake's Impact Damage Potential

### Load The Dataset

In [1]:
import pandas as pd

df = pd.read_csv('C:/Users/divyanshu/OneDrive/Desktop/Projects/ImpactSense/impactsenseAI-Infosys-Intern-project/data/preprocessed_earthquake_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109848 entries, 0 to 109847
Data columns (total 15 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   latitude   109848 non-null  float64
 1   longitude  109848 non-null  float64
 2   depth      109848 non-null  float64
 3   mag        109848 non-null  float64
 4   magType    109848 non-null  object 
 5   rms        109848 non-null  float64
 6   type       109848 non-null  object 
 7   status     109848 non-null  object 
 8   Year       109848 non-null  int64  
 9   Month      109848 non-null  int64  
 10  Day        109848 non-null  int64  
 11  Hour       109848 non-null  int64  
 12  Minute     109848 non-null  int64  
 13  Second     109848 non-null  int64  
 14  DayOfWeek  109848 non-null  int64  
dtypes: float64(5), int64(7), object(3)
memory usage: 12.6+ MB


### 1. Magnitude Type Prioritization and Standardization

In [2]:
def convert_to_mw(mag_value, mag_type):
    """Convert various earthquake magnitude types to moment magnitude (Mw)."""
    # Reference conversions based on empirical relationships
    if mag_type.lower().startswith('mw'):
        # Moment Magnitude is already in the desired format
        return mag_value
    elif mag_type.lower() == 'ms':
        # Surface Wave Magnitude to Moment Magnitude
        return 1.05 * mag_value - 0.2
    elif mag_type.lower() == 'mb':
        # Body Wave Magnitude to Moment Magnitude
        if mag_value > 6.5:
            return 6.5 + (mag_value - 6.5) * 1.5 # Correction for saturation
        # For mb <= 6.5
        return 0.67 * mag_value + 3.2
    # Add more conversions as needed
    elif mag_type.lower() == 'ml':
        return 1.2 * mag_value - 1.0
    else:
        # Unknown magnitude type
        return np.nan

In [3]:
# Apply the conversion to the DataFrame
import numpy as np
df['Mw'] = df.apply(lambda row: convert_to_mw(row['mag'], row['magType']), axis=1)
df[['mag', 'magType', 'Mw']].head(10)

,mag,magType,Mw
0,3.296720,mw,3.296720
1,2.920854,mw,2.920854
2,4.716657,mw,4.716657
3,4.299028,mw,4.299028
4,3.442890,mw,3.442890
5,3.860518,mw,3.860518
6,2.670277,mw,2.670277
7,2.190004,mw,2.190004
8,2.712040,mw,2.712040
9,2.231767,mw,2.231767


#### 2. Calculate Damage Potential Using HAZUS-Style Formula

In [4]:
def calculate_damage_potential_hazus(magnitude, depth):
    """Calculate earthquake damage potential using HAZUS methodology."""
    actual_depth = max(abs(depth), 1.0)  # Ensure depth is at least 1 km to avoid log(0)
    log_pga = magnitude - 3.5 * np.log10(actual_depth + 7) + 1.8 # HAZUS empirical formula
    pga = 10 ** log_pga  # Convert log10(PGA) to PGA in g
    # Calculate damage potential score (0 to 10 scale)
    damage_potential = min(10.0, max(0.0, 2.5 * np.log10(pga + 0.01) + 7.5))
    # Return the final damage potential score
    return damage_potential

In [5]:
# Apply the conversion to the DataFrame
df['damage_potential'] = df.apply(lambda row: calculate_damage_potential_hazus(row['Mw'], row['depth']), axis=1)
df[['Mw', 'depth', 'damage_potential']].head(10)

,Mw,depth,damage_potential
0,3.296720,-0.430517,10.000000
1,2.920854,-0.430517,10.000000
2,4.716657,-0.291162,10.000000
3,4.299028,-0.430517,10.000000
4,3.442890,-0.430517,10.000000
5,3.860518,-0.476968,10.000000
6,2.670277,-0.476968,10.000000
7,2.190004,-0.430517,9.574581
8,2.712040,-0.384065,10.000000
9,2.231767,-0.430517,9.678841
